https://github.com/heli2305/TriTueNhanTao

In [1]:
import tkinter as tk
from tkinter import ttk
from collections import deque
from dataclasses import dataclass
import heapq
import random

# Du lieu ban dau
SIZE = 5
START_GRID = (
    (0, 0, 0, 0, 0),
    (0, 1, 1, 0, 0),
    (0, 0, 0, 1, 0),
    (0, 1, 0, 0, 0),
    (0, 0, 0, 1, 1),
)
START_POS = (0, 0)
MAX_RESTART = 5
MAX_SHOW_RECORDS = 300
MAX_SEARCH_RECORDS = 10000
MAX_FRONTIER_SHOW = 60
MAX_IDS_DEPTH = 30
STEP_DELAY = 250
GOAL_GRID = (
    (0, 0, 0, 0, 0),
    (0, 0, 0, 0, 0),
    (0, 0, 0, 0, 0),
    (0, 0, 0, 0, 0),
    (0, 0, 0, 0, 0),
)


@dataclass
class Node:
    name: str
    grid: tuple
    pos: tuple
    parent: object = None
    action: str = ""
    cost: int = 0


# Xu ly tim kiem
def is_goal(grid):
    for row in grid:
        for cell in row:
            if cell != 0:
                return False
    return True


def state_key(node):
    return (node.grid, node.pos)


def heuristic(node):
    dirty_cells = []

    for r in range(SIZE):
        for c in range(SIZE):
            if node.grid[r][c] == 1:
                dirty_cells.append((r, c))

    if not dirty_cells:
        return 0

    robot_r, robot_c = node.pos
    min_distance = SIZE * SIZE

    for dirty_r, dirty_c in dirty_cells:
        distance = abs(robot_r - dirty_r) + abs(robot_c - dirty_c)
        if distance < min_distance:
            min_distance = distance

    return len(dirty_cells) + min_distance


def next_name(i):
    letters = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
    if i < len(letters):
        return letters[i]
    return letters[i % len(letters)] + str(i // len(letters))


def valid_moves(pos):
    r, c = pos
    moves = []

    if r > 0:
        moves.append(("U", -1, 0))
    if r < SIZE - 1:
        moves.append(("D", 1, 0))
    if c > 0:
        moves.append(("L", 0, -1))
    if c < SIZE - 1:
        moves.append(("R", 0, 1))
    return moves


def make_child(node, move, name):
    action, dr, dc = move
    r, c = node.pos

    grid = [list(row) for row in node.grid]

    # Neu dang o o do thi hut sach truoc khi di chuyen
    if grid[r][c] == 1:
        grid[r][c] = 0

    nr, nc = r + dr, c + dc
    new_grid = tuple(tuple(row) for row in grid)

    return Node(
        name=name,
        grid=new_grid,
        pos=(nr, nc),
        parent=node,
        action=action,
        cost=node.cost + 1
    )


def path_to_root(node):
    path = []
    while node is not None:
        path.append(node)
        node = node.parent
    path.reverse()
    return path


def matrix_text(grid, pos=None):
    lines = []
    for r in range(SIZE):
        row = []
        for c in range(SIZE):
            if pos == (r, c):
                row.append("x")
            else:
                row.append(str(grid[r][c]))
        lines.append(" ".join(row))
    return "\n".join(lines)


def matrix_short(grid, pos=None):
    rows = []
    for r in range(SIZE):
        row = []
        for c in range(SIZE):
            if pos == (r, c):
                row.append("x")
            else:
                row.append(str(grid[r][c]))
        rows.append("".join(row))
    return "/".join(rows)


def is_cycle(node):
    current_key = state_key(node)
    parent = node.parent

    while parent is not None:
        if state_key(parent) == current_key:
            return True
        parent = parent.parent

    return False


def depth_limited_search(limit, version=1, record_limit=MAX_SEARCH_RECORDS):
    start = Node("A", START_GRID, START_POS)
    frontier = [start]
    records = []
    expanded_names = []
    name_index = 1
    result = "failure"

    def short_frontier():
        return list(frontier[-MAX_FRONTIER_SHOW:])

    def reached_short():
        return list(expanded_names[-MAX_FRONTIER_SHOW:])

    def over_record_limit(node):
        records.append({
            "node_label": node.name,
            "show_node": node,
            "frontier": short_frontier(),
            "reached": reached_short(),
            "note": "Dung IDS vi qua nhieu buoc, tranh treo giao dien"
        })
        return "search_cutoff", records

    if is_goal(start.grid):
        records.append({
            "node_label": "A",
            "show_node": start,
            "frontier": [],
            "reached": [],
            "note": "Trang thai dau da la G"
        })
        return start, records

    while frontier:
        node = frontier.pop()
        expanded_names.append(node.name)

        if len(records) >= record_limit:
            return over_record_limit(node)

        if version == 1 and is_goal(node.grid):
            records.append({
                "node_label": node.name,
                "show_node": node,
                "frontier": short_frontier(),
                "reached": reached_short(),
                "note": "Tim thay G"
            })
            return node, records

        if node.cost >= limit:
            result = "cutoff"
            records.append({
                "node_label": node.name,
                "show_node": node,
                "frontier": short_frontier(),
                "reached": reached_short(),
                "note": f"Cham gioi han do sau {limit}"
            })
            continue

        if is_cycle(node):
            records.append({
                "node_label": node.name,
                "show_node": node,
                "frontier": short_frontier(),
                "reached": reached_short(),
                "note": "Bo qua vi tao chu trinh"
            })
            continue

        found_child = None

        for move in reversed(valid_moves(node.pos)):
            child = make_child(node, move, next_name(name_index))
            name_index += 1

            if version == 2 and is_goal(child.grid):
                found_child = child
                break

            frontier.append(child)

        if found_child is not None:
            records.append({
                "node_label": f"{node.name} -> {found_child.name}",
                "show_node": found_child,
                "frontier": short_frontier(),
                "reached": reached_short(),
                "note": f"Tim thay {found_child.name} khi sinh tu {node.name}"
            })
            return found_child, records

        records.append({
            "node_label": node.name,
            "show_node": node,
            "frontier": short_frontier(),
            "reached": reached_short(),
            "note": "Da mo rong node"
        })

    return result, records


def depth_limited_search_for_ids(limit, version=1):
    start = Node("A", START_GRID, START_POS)
    frontier = [start]
    name_index = 1
    expanded_count = 0

    while frontier:
        node = frontier.pop()
        expanded_count += 1

        if version == 1 and is_goal(node.grid):
            return node, expanded_count

        if node.cost >= limit:
            continue

        if is_cycle(node):
            continue

        for move in reversed(valid_moves(node.pos)):
            child = make_child(node, move, next_name(name_index))
            name_index += 1

            if version == 2 and is_goal(child.grid):
                return child, expanded_count

            frontier.append(child)

    return None, expanded_count


def iterative_deepening_search(version=1):
    all_records = []
    start = Node("A", START_GRID, START_POS)

    for depth in range(MAX_IDS_DEPTH + 1):
        result, expanded_count = depth_limited_search_for_ids(depth, version=version)

        record = {
            "node_label": f"d={depth}",
            "show_node": start,
            "frontier": [],
            "reached": [f"{expanded_count} node"],
            "note": f"IDS cach {version}: da thu depth {depth}, mo rong {expanded_count} node"
        }
        record["reset_frontier_names"] = True
        record["section"] = f"IDS cach {version} - depth = {depth}"
        all_records.append(record)

        if isinstance(result, Node):
            path = path_to_root(result)
            path_names = []
            for node in path:
                path_names.append(node.name)
                all_records.append({
                    "node_label": node.name,
                    "show_node": node,
                    "frontier": [],
                    "reached": list(path_names),
                    "note": "Node nam tren duong di ket qua cua IDS"
                })
            return result, all_records, "success"

    return None, all_records, "cutoff"


def uniform_cost_search():
    start = Node("A", START_GRID, START_POS)
    frontier = [(start.cost, 0, start)]
    best_costs = {state_key(start): start.cost}
    records = []
    reached_names = []
    name_index = 1
    push_index = 1

    if is_goal(start.grid):
        records.append({
            "node_label": "A",
            "show_node": start,
            "frontier": [],
            "reached": [],
            "note": "Trang thai dau da la G"
        })
        return start, records, "success"

    while frontier:
        _, _, node = heapq.heappop(frontier)
        node_key = state_key(node)

        if node.cost > best_costs.get(node_key, float("inf")):
            continue

        reached_names.append(node.name)

        if is_goal(node.grid):
            records.append({
                "node_label": node.name,
                "show_node": node,
                "frontier": [item[2] for item in sorted(frontier)],
                "reached": list(reached_names),
                "note": "Tim thay G"
            })
            return node, records, "success"

        for move in valid_moves(node.pos):
            child = make_child(node, move, next_name(name_index))
            name_index += 1
            child_key = state_key(child)

            if child.cost < best_costs.get(child_key, float("inf")):
                best_costs[child_key] = child.cost
                heapq.heappush(frontier, (child.cost, push_index, child))
                push_index += 1

        records.append({
            "node_label": node.name,
            "show_node": node,
            "frontier": [item[2] for item in sorted(frontier)],
            "reached": list(reached_names),
            "note": "Da mo rong node"
        })

    return None, records, "failure"


def greedy_search():
    start = Node("A", START_GRID, START_POS)
    frontier = [start]
    reached_keys = set()
    records = []
    reached_names = []
    name_index = 1

    if is_goal(start.grid):
        records.append({
            "node_label": "A",
            "show_node": start,
            "frontier": [],
            "reached": [],
            "note": "Trang thai dau da la G"
        })
        return start, records, "success"

    while frontier:
        best_index = 0
        for i in range(1, len(frontier)):
            if heuristic(frontier[i]) < heuristic(frontier[best_index]):
                best_index = i

        node = frontier.pop(best_index)
        reached_keys.add(state_key(node))
        reached_names.append(node.name)

        if is_goal(node.grid):
            records.append({
                "node_label": node.name,
                "show_node": node,
                "frontier": sorted(frontier, key=heuristic),
                "reached": list(reached_names),
                "note": "Tim thay G"
            })
            return node, records, "success"

        for move in valid_moves(node.pos):
            child = make_child(node, move, next_name(name_index))
            child_key = state_key(child)

            in_frontier = False
            for item in frontier:
                if state_key(item) == child_key:
                    in_frontier = True
                    break

            if child_key in reached_keys or in_frontier:
                continue

            name_index += 1
            frontier.append(child)

        records.append({
            "node_label": node.name,
            "show_node": node,
            "frontier": sorted(frontier, key=heuristic),
            "reached": list(reached_names),
            "note": f"Da mo rong node, h = {heuristic(node)}"
        })

    return None, records, "failure"


def a_star_search():
    start = Node("A", START_GRID, START_POS)
    frontier = [start]
    reached = {}
    records = []
    reached_names = []
    name_index = 1

    if is_goal(start.grid):
        records.append({
            "node_label": "A",
            "show_node": start,
            "frontier": [],
            "reached": [],
            "note": "Trang thai dau da la G"
        })
        return start, records, "success"

    while frontier:
        # Chon node co f = g + h nho nhat trong FRONTIER
        best_index = 0
        for i in range(1, len(frontier)):
            f_i = frontier[i].cost + heuristic(frontier[i])
            f_best = frontier[best_index].cost + heuristic(frontier[best_index])
            if f_i < f_best:
                best_index = i

        node = frontier.pop(best_index)

        if is_goal(node.grid):
            records.append({
                "node_label": node.name,
                "show_node": node,
                "frontier": sorted(frontier, key=lambda item: item.cost + heuristic(item)),
                "reached": list(reached_names),
                "note": "Tim thay G"
            })
            return node, records, "success"

        # Dua node vua xet vao REACHED
        reached[state_key(node)] = node
        reached_names.append(node.name)

        for move in valid_moves(node.pos):
            child = make_child(node, move, next_name(name_index))
            child_key = state_key(child)
            old_reached = reached.get(child_key)

            # Neu da co trong REACHED ma duong moi khong tot hon thi bo qua
            if old_reached is not None:
                if child.cost >= old_reached.cost:
                    continue
                del reached[child_key]
                if old_reached.name in reached_names:
                    reached_names.remove(old_reached.name)

            # Neu da co trong FRONTIER thi chi cap nhat khi g moi nho hon
            frontier_index = -1
            for i in range(len(frontier)):
                if state_key(frontier[i]) == child_key:
                    frontier_index = i
                    break

            if frontier_index != -1:
                if child.cost < frontier[frontier_index].cost:
                    child.name = frontier[frontier_index].name
                    frontier[frontier_index] = child
                continue

            name_index += 1
            frontier.append(child)

        records.append({
            "node_label": node.name,
            "show_node": node,
            "frontier": sorted(frontier, key=lambda item: item.cost + heuristic(item)),
            "reached": list(reached_names),
            "note": f"Da mo rong node, f = {node.cost + heuristic(node)}"
        })

    return None, records, "failure"


def bfs_dfs_search(method, version=1):
    start = Node("A", START_GRID, START_POS)
    records = []
    reached_keys = set()
    reached_names = []
    name_index = 1

    if method == "BFS":
        frontier = deque([start])
    else:
        frontier = [start]

    if is_goal(start.grid):
        records.append({
            "node_label": "A",
            "show_node": start,
            "frontier": [],
            "reached": [],
            "note": "Trang thai dau da la G"
        })
        return start, records, "success"

    while frontier:
        node = frontier.popleft() if method == "BFS" else frontier.pop()
        reached_keys.add(state_key(node))
        reached_names.append(node.name)

        if version == 1 and is_goal(node.grid):
            records.append({
                "node_label": node.name,
                "show_node": node,
                "frontier": list(frontier),
                "reached": list(reached_names),
                "note": "Tim thay G"
            })
            return node, records, "success"

        found_child = None
        moves = valid_moves(node.pos)

        if method == "DFS":
            moves = list(reversed(moves))

        for move in moves:
            child = make_child(node, move, next_name(name_index))
            child_key = state_key(child)

            in_frontier = False
            for item in frontier:
                if state_key(item) == child_key:
                    in_frontier = True
                    break

            if child_key in reached_keys or in_frontier:
                continue

            name_index += 1

            if version == 2 and is_goal(child.grid):
                found_child = child
                break

            frontier.append(child)

        if found_child is not None:
            records.append({
                "node_label": f"{node.name} -> {found_child.name}",
                "show_node": found_child,
                "frontier": list(frontier),
                "reached": list(reached_names),
                "note": f"Tim thay {found_child.name} khi sinh tu {node.name}"
            })
            return found_child, records, "success"

        records.append({
            "node_label": node.name,
            "show_node": node,
            "frontier": list(frontier),
            "reached": list(reached_names),
            "note": "Da mo rong node"
        })

    return None, records, "failure"


def simple_hill_climbing():
    start = Node("A", START_GRID, START_POS)
    records = []
    reached_names = [start.name]
    name_index = 1
    current = start
    
    def value_func(n):
        return -heuristic(n)
        
    while True:
        if is_goal(current.grid):
            records.append({
                "node_label": current.name,
                "show_node": current,
                "frontier": [],
                "reached": list(reached_names),
                "note": "Tim thay G"
            })
            return current, records, "success"
            
        neighbors = []
        for move in valid_moves(current.pos):
            child = make_child(current, move, next_name(name_index))
            name_index += 1
            neighbors.append(child)
            
        found_better = False
        next_state = None
        for neighbor in neighbors:
            if value_func(neighbor) > value_func(current):
                next_state = neighbor
                found_better = True
                break
                
        records.append({
            "node_label": current.name,
            "show_node": current,
            "frontier": list(neighbors),
            "reached": list(reached_names),
            "note": f"Dang o node {current.name}, v = {value_func(current)}. " + 
                    (f"Di den {next_state.name}" if found_better else "Ket thuc tai cuc dai cuc bo")
        })
        
        if found_better:
            reached_names.append(next_state.name)
            current = next_state
        else:
            return current, records, "failure"


def steepest_ascent_hill_climbing():
    start = Node("A", START_GRID, START_POS)
    records = []
    reached_names = [start.name]
    name_index = 1
    current = start
    
    def value_func(n):
        return -heuristic(n)
        
    while True:
        if is_goal(current.grid):
            records.append({
                "node_label": current.name,
                "show_node": current,
                "frontier": [],
                "reached": list(reached_names),
                "note": "Tim thay G"
            })
            return current, records, "success"
            
        neighbors = []
        for move in valid_moves(current.pos):
            child = make_child(current, move, next_name(name_index))
            name_index += 1
            neighbors.append(child)
            
        if not neighbors:
            records.append({
                "node_label": current.name,
                "show_node": current,
                "frontier": [],
                "reached": list(reached_names),
                "note": "Khong co trang thai lan can"
            })
            return current, records, "failure"
            
        best_neighbor = max(neighbors, key=value_func)
        found_better = value_func(best_neighbor) > value_func(current)
        
        records.append({
            "node_label": current.name,
            "show_node": current,
            "frontier": list(neighbors),
            "reached": list(reached_names),
            "note": f"Dang o node {current.name}, v = {value_func(current)}. " + 
                    (f"Di den Best_Neighbor {best_neighbor.name}" if found_better else "Ket thuc tai cuc dai cuc bo")
        })
        
        if found_better:
            reached_names.append(best_neighbor.name)
            current = best_neighbor
        else:
            return current, records, "failure"


def stochastic_hill_climbing():
    start = Node("A", START_GRID, START_POS)
    records = []
    reached_names = [start.name]
    name_index = 1
    current = start
    
    def value_func(n):
        return -heuristic(n)
        
    while True:
        if is_goal(current.grid):
            records.append({
                "node_label": current.name,
                "show_node": current,
                "frontier": [],
                "reached": list(reached_names),
                "note": "Tim thay G"
            })
            return current, records, "success"
            
        neighbors = []
        for move in valid_moves(current.pos):
            child = make_child(current, move, next_name(name_index))
            name_index += 1
            neighbors.append(child)
            
        better_neighbors = [neighbor for neighbor in neighbors if value_func(neighbor) > value_func(current)]
        
        if not better_neighbors:
            records.append({
                "node_label": current.name,
                "show_node": current,
                "frontier": list(neighbors),
                "reached": list(reached_names),
                "note": f"Dang o node {current.name}, v = {value_func(current)}. Ket thuc tai cuc dai cuc bo (Better_Neighbors rong)"
            })
            return current, records, "failure"
        else:
            next_state = random.choice(better_neighbors)
            records.append({
                "node_label": current.name,
                "show_node": current,
                "frontier": list(neighbors),
                "reached": list(reached_names),
                "note": f"Dang o node {current.name}, v = {value_func(current)}. Chon ngau nhien {next_state.name} tu {len(better_neighbors)} node tot hon"
            })
            reached_names.append(next_state.name)
            current = next_state


def random_restart_hill_climbing():
    records = []
    name_index = 1
    best_node = None

    def value_func(n):
        return -heuristic(n)

    for restart in range(1, MAX_RESTART + 1):
        # Luot dau dung S, cac luot sau tao lai vi tri robot ngau nhien
        if restart == 1:
            current = Node("A", START_GRID, START_POS)
        else:
            random_pos = (random.randrange(SIZE), random.randrange(SIZE))
            current = Node(next_name(name_index), START_GRID, random_pos)
            name_index += 1

        reached_names = [current.name]
        first_record = True

        while True:
            if best_node is None or value_func(current) > value_func(best_node):
                best_node = current

            if is_goal(current.grid):
                record = {
                    "node_label": current.name,
                    "show_node": current,
                    "frontier": [],
                    "reached": list(reached_names),
                    "note": "Tim thay G"
                }
                if first_record:
                    record["section"] = f"Restart {restart}/{MAX_RESTART}"
                    record["reset_frontier_names"] = True
                records.append(record)
                return current, records, "success"

            neighbors = []
            for move in valid_moves(current.pos):
                child = make_child(current, move, next_name(name_index))
                name_index += 1
                neighbors.append(child)

            # Better_Neighbors gom cac trang thai co value tot hon current
            better_neighbors = []
            for neighbor in neighbors:
                if value_func(neighbor) > value_func(current):
                    better_neighbors.append(neighbor)

            if not better_neighbors:
                note = f"Dang o node {current.name}, v = {value_func(current)}. Better_Neighbors rong"
                if restart < MAX_RESTART:
                    note += ", chuyen sang restart tiep theo"
                else:
                    note += ", da het MAX_RESTART"

                record = {
                    "node_label": current.name,
                    "show_node": current,
                    "frontier": [],
                    "reached": list(reached_names),
                    "note": note
                }
                if first_record:
                    record["section"] = f"Restart {restart}/{MAX_RESTART}"
                    record["reset_frontier_names"] = True
                records.append(record)
                break

            # Chon trang thai tot nhat trong Better_Neighbors
            next_state = max(better_neighbors, key=value_func)
            record = {
                "node_label": current.name,
                "show_node": current,
                "frontier": list(better_neighbors),
                "reached": list(reached_names),
                "note": f"Dang o node {current.name}, v = {value_func(current)}. Chon Best_Neighbor {next_state.name}"
            }
            if first_record:
                record["section"] = f"Restart {restart}/{MAX_RESTART}"
                record["reset_frontier_names"] = True
                first_record = False
            records.append(record)

            reached_names.append(next_state.name)
            current = next_state

    return best_node, records, "failure"


def f_limited_search(f_limit, record_limit):
    start = Node("A", START_GRID, START_POS)
    frontier = [start]
    records = []
    expanded_names = []
    name_index = 1
    next_limit = float("inf")

    def short_frontier():
        return list(frontier[-MAX_FRONTIER_SHOW:])

    def reached_short():
        return list(expanded_names[-MAX_FRONTIER_SHOW:])

    def over_record_limit(node, note):
        records.append({
            "node_label": node.name,
            "show_node": node,
            "frontier": short_frontier(),
            "reached": reached_short(),
            "note": note
        })
        return "cutoff", records, next_limit
    
    if is_goal(start.grid):
        records.append({
            "node_label": "A",
            "show_node": start,
            "frontier": [],
            "reached": [],
            "note": "Trang thai dau da la G"
        })
        return start, records, next_limit
        
    while frontier:
        node = frontier.pop()
        expanded_names.append(node.name)

        if len(records) >= record_limit:
            return over_record_limit(node, "Dung IDA* vi qua nhieu buoc, tranh treo giao dien")
        
        f_value = node.cost + heuristic(node)
        
        if f_value > f_limit:
            next_limit = min(next_limit, f_value)
            records.append({
                "node_label": node.name,
                "show_node": node,
                "frontier": short_frontier(),
                "reached": reached_short(),
                "note": f"Vuot gioi han f: f = {f_value} > {f_limit}"
            })
            continue
            
        if is_goal(node.grid):
            records.append({
                "node_label": node.name,
                "show_node": node,
                "frontier": short_frontier(),
                "reached": reached_short(),
                "note": "Tim thay G"
            })
            return node, records, next_limit
            
        if is_cycle(node):
            records.append({
                "node_label": node.name,
                "show_node": node,
                "frontier": short_frontier(),
                "reached": reached_short(),
                "note": "Bo qua vi tao chu trinh"
            })
            continue
            
        for move in reversed(valid_moves(node.pos)):
            child = make_child(node, move, next_name(name_index))
            name_index += 1
            frontier.append(child)
            
        records.append({
            "node_label": node.name,
            "show_node": node,
            "frontier": short_frontier(),
            "reached": reached_short(),
            "note": f"Da mo rong node, f = {f_value}"
        })
        
    return None, records, next_limit


def ida_star_search():
    start = Node("A", START_GRID, START_POS)
    f_limit = start.cost + heuristic(start)
    all_records = []
    
    while f_limit != float("inf"):
        remaining_records = MAX_SEARCH_RECORDS - len(all_records)
        if remaining_records <= 0:
            return None, all_records, "cutoff"

        result, records, next_limit = f_limited_search(f_limit, remaining_records)
        
        if records:
            records[0]["reset_frontier_names"] = True
            records[0]["section"] = f"IDA* - f_limit = {f_limit}"
            all_records.extend(records)
            
        if isinstance(result, Node):
            return result, all_records, "success"

        if result == "cutoff":
            return None, all_records, "cutoff"
            
        if next_limit == float("inf"):
            break
            
        f_limit = next_limit
        
    return None, all_records, "failure"


def search(method, version=1):
    if method == "UCS":
        return uniform_cost_search()

    if method == "Greedy":
        return greedy_search()

    if method == "A*":
        return a_star_search()

    if method == "IDS":
        return iterative_deepening_search(version=version)

    if method == "IDA*":
        return ida_star_search()

    if method == "Simple Hill Climbing":
        return simple_hill_climbing()

    if method == "Steepest Ascent Hill":
        return steepest_ascent_hill_climbing()

    if method == "Stochastic Hill":
        return stochastic_hill_climbing()

    if method == "Random Restart Hill":
        return random_restart_hill_climbing()

    return bfs_dfs_search(method, version=version)


# Giao dien

class VacuumApp(tk.Tk):
    def __init__(self):
        super().__init__()

        self.title("May hut bui - BFS, DFS, UCS, IDS, Greedy, A*, IDA*, Hill Climbing")
        self.geometry("1440x800")
        self.minsize(1180, 680)
        self.configure(bg="#f2f4f7")

        self.records = []
        self.display_records = []
        self.was_record_limited = False
        self.goal_node = None
        self.search_status = "failure"
        self.current_title = ""
        self.after_id = None
        self.shown_frontier_names = set()

        self.make_ui()
        self.draw_start_screen()

    def make_ui(self):
        style = ttk.Style()
        style.theme_use("clam")
        style.configure("TButton", font=("Arial", 10, "bold"), padding=(8, 8))

        main = tk.Frame(self, bg="#f2f4f7", padx=16, pady=16)
        main.pack(fill="both", expand=True)

        main.grid_columnconfigure(1, weight=2)
        main.grid_columnconfigure(2, weight=0, minsize=520)
        main.grid_rowconfigure(0, weight=1)

        # Cot nut
        left = tk.Frame(main, bg="#f2f4f7")
        left.grid(row=0, column=0, sticky="ns", padx=(0, 16))

        ttk.Button(left, text="BFS cách 1", command=lambda: self.run_algo("BFS", 1)).pack(fill="x", pady=4)
        ttk.Button(left, text="BFS cách 2", command=lambda: self.run_algo("BFS", 2)).pack(fill="x", pady=4)
        ttk.Button(left, text="DFS cách 1", command=lambda: self.run_algo("DFS", 1)).pack(fill="x", pady=4)
        ttk.Button(left, text="DFS cách 2", command=lambda: self.run_algo("DFS", 2)).pack(fill="x", pady=4)
        ttk.Button(left, text="UCS", command=lambda: self.run_algo("UCS")).pack(fill="x", pady=4)
        ttk.Button(left, text="Greedy", command=lambda: self.run_algo("Greedy")).pack(fill="x", pady=4)
        ttk.Button(left, text="A*", command=lambda: self.run_algo("A*")).pack(fill="x", pady=4)
        ttk.Button(left, text="IDS cách 1", command=lambda: self.run_algo("IDS", 1)).pack(fill="x", pady=4)
        ttk.Button(left, text="IDS cách 2", command=lambda: self.run_algo("IDS", 2)).pack(fill="x", pady=4)
        ttk.Button(left, text="IDA*", command=lambda: self.run_algo("IDA*")).pack(fill="x", pady=4)
        ttk.Button(left, text="Simple Hill Climbing", command=lambda: self.run_algo("Simple Hill Climbing")).pack(fill="x", pady=4)
        ttk.Button(left, text="Steepest Ascent Hill", command=lambda: self.run_algo("Steepest Ascent Hill")).pack(fill="x", pady=4)
        ttk.Button(left, text="Stochastic Hill", command=lambda: self.run_algo("Stochastic Hill")).pack(fill="x", pady=4)
        ttk.Button(left, text="Random Restart Hill", command=lambda: self.run_algo("Random Restart Hill")).pack(fill="x", pady=4)

        # Cot giua
        middle = tk.Frame(main, bg="#f2f4f7")
        middle.grid(row=0, column=1, sticky="nsew")
        middle.grid_columnconfigure(0, weight=1)
        middle.grid_rowconfigure(0, weight=5)
        middle.grid_rowconfigure(1, weight=0, minsize=240)

        screen_box = tk.LabelFrame(
            middle,
            text="Màn hình minh họa hoạt động",
            font=("Arial", 12, "bold"),
            bg="white",
            padx=8,
            pady=8
        )
        screen_box.grid(row=0, column=0, sticky="nsew")
        screen_box.grid_columnconfigure(0, weight=1)
        screen_box.grid_rowconfigure(0, weight=1)

        self.canvas = tk.Canvas(screen_box, bg="white", highlightthickness=0)
        self.canvas.grid(row=0, column=0, sticky="nsew")

        result_box = tk.LabelFrame(
            middle,
            text="Kết quả chạy",
            font=("Arial", 12, "bold"),
            bg="white",
            height=240,
            padx=8,
            pady=8
        )
        result_box.grid(row=1, column=0, sticky="nsew", pady=(10, 0))
        result_box.grid_propagate(False)
        result_box.grid_columnconfigure(0, weight=1)
        result_box.grid_rowconfigure(0, weight=1)

        self.result_text = tk.Text(
            result_box,
            font=("Consolas", 10),
            wrap="word",
            state="disabled",
            bg="white",
            relief="flat"
        )
        self.result_text.grid(row=0, column=0, sticky="nsew")

        # Cot phai
        process_box = tk.LabelFrame(
            main,
            text="Quá trình các bước chạy",
            font=("Arial", 12, "bold"),
            bg="white",
            width=480,
            padx=8,
            pady=8
        )
        process_box.grid(row=0, column=2, sticky="nsew", padx=(16, 0))
        process_box.grid_propagate(False)
        process_box.grid_columnconfigure(0, weight=1)
        process_box.grid_rowconfigure(0, weight=1)

        self.process_text = tk.Text(
            process_box,
            width=62,
            font=("Consolas", 10),
            wrap="none",
            state="disabled",
            bg="white"
        )
        self.process_text.grid(row=0, column=0, sticky="nsew")

        yscroll = ttk.Scrollbar(process_box, orient="vertical", command=self.process_text.yview)
        yscroll.grid(row=0, column=1, sticky="ns")
        self.process_text.configure(yscrollcommand=yscroll.set)

    def draw_start_screen(self):
        self.canvas.delete("all")
        self.set_result(
            "S =\n" + matrix_text(START_GRID, START_POS) +
            "\n\nG =\n" + matrix_text(GOAL_GRID)
        )
        # Render the start state immediately on startup!
        start_node = Node("A", START_GRID, START_POS)
        self.draw_state(start_node, "Trạng thái bắt đầu", "Bấm nút bên trái để chạy thuật toán")

    def set_result(self, text):
        self.result_text.config(state="normal")
        self.result_text.delete("1.0", "end")
        self.result_text.insert("end", text)
        self.result_text.config(state="disabled")

    def clear_process(self):
        self.shown_frontier_names = set()
        self.process_text.config(state="normal")
        self.process_text.delete("1.0", "end")
        if "Hill" in self.current_title:
            self.process_text.insert(
                "end",
                f'{"Current":<10}| Neighbors\n' +
                "-" * 10 + "+-" + "-" * 70 + "\n"
            )
        else:
            self.process_text.insert(
                "end",
                f'{"Node":<8}| {"Frontier":<42}| Reached\n' +
                "-" * 8 + "+-" + "-" * 42 + "+-" + "-" * 18 + "\n"
            )
        self.process_text.config(state="disabled")

    def run_algo(self, method, version=1):
        if self.after_id is not None:
            self.after_cancel(self.after_id)
            self.after_id = None

        if method in {"BFS", "DFS", "IDS"}:
            self.current_title = f"{method} cách {version}"
            self.goal_node, self.records, self.search_status = search(method, version=version)
        elif method in {"UCS", "Greedy", "A*", "IDA*", "Simple Hill Climbing", "Steepest Ascent Hill", "Stochastic Hill", "Random Restart Hill"}:
            self.current_title = method
            self.goal_node, self.records, self.search_status = search(method)

        self.clear_process()
        self.set_result("Đang tính toán lời giải bằng " + self.current_title + "...")

        self.display_records = self.records[:MAX_SHOW_RECORDS]
        self.was_record_limited = len(self.records) > MAX_SHOW_RECORDS

        for record in self.display_records:
            self.add_process_row(record)

        if self.was_record_limited:
            self.process_text.config(state="normal")
            self.process_text.insert("end", f"\nChi hien thi {MAX_SHOW_RECORDS} buoc dau de tranh dung giao dien.\n")
            self.process_text.config(state="disabled")

        if self.current_title == "Random Restart Hill" and self.display_records:
            self.show_random_restart_step(0)
        elif self.goal_node is not None and isinstance(self.goal_node, Node):
            self.solution_path = path_to_root(self.goal_node)
            self.show_solution_step(0)
        else:
            self.show_final_result()

    def add_process_row(self, record):
        if record.get("reset_frontier_names"):
            self.shown_frontier_names = set()

        self.process_text.config(state="normal")
        if record.get("section"):
            self.process_text.insert("end", f"[{record['section']}]\n")
        self.process_text.config(state="disabled")

        frontier_lines = []

        for node in record["frontier"]:
            if node.name in self.shown_frontier_names and "A*" not in self.current_title:
                frontier_lines.append(node.name)
            else:
                parent_name = node.parent.name if node.parent else "-"
                action_name = node.action if node.action else "-"
                
                # Dynamic indicator formatting based on algorithm
                if "A*" in self.current_title:
                    f_value = node.cost + heuristic(node)
                    details = f"cost={node.cost}, h={heuristic(node)}, f={f_value}"
                elif "Greedy" in self.current_title:
                    details = f"cost={node.cost}, h={heuristic(node)}"
                elif "Hill" in self.current_title:
                    details = f"h={heuristic(node)}, v={-heuristic(node)}"
                else:
                    details = f"cost={node.cost}"
                
                line = f"{node.name}: [{matrix_short(node.grid, node.pos)}], {parent_name}, {action_name}, {details}"
                frontier_lines.append(line)
                self.shown_frontier_names.add(node.name)

        if not frontier_lines:
            frontier_lines = ["(rỗng)"]

        self.process_text.config(state="normal")
        if "Hill" in self.current_title:
            for i, line in enumerate(frontier_lines):
                current_label = record["node_label"] if i == 0 else ""
                self.process_text.insert("end", f"{current_label:<10}| {line}\n")
        else:
            reached_text = "{" + ", ".join(record["reached"]) + "}" if record["reached"] else "{}"
            for i, line in enumerate(frontier_lines):
                node_label = record["node_label"] if i == 0 else ""
                reached = reached_text if i == 0 else ""
                self.process_text.insert("end", f"{node_label:<8}| {line:<42}| {reached}\n")
        self.process_text.insert("end", "\n")
        self.process_text.see("end")
        self.process_text.config(state="disabled")

    def draw_grid(self, x, y, cell, grid, pos, label):
        self.canvas.create_text(
            x + cell * (SIZE / 2.0), y - 18, 
            text=label, 
            font=("Arial", 13, "bold"), 
            fill="#1e293b"
        )

        for r in range(SIZE):
            for c in range(SIZE):
                left = x + c * cell
                top = y + r * cell
                value = grid[r][c]

                if pos == (r, c):
                    fill = "#ffe4e6" if value == 1 else "#dbeafe" 
                else:
                    fill = "#fef3c7" if value == 1 else "#f1f5f9" 

                self.canvas.create_rectangle(
                    left, top, left + cell, top + cell,
                    fill=fill, outline="#64748b", width=2
                )

                if pos == (r, c):
                    center_x = left + cell / 2
                    center_y = top + cell / 2
                    radius = cell * 0.34

                    self.canvas.create_oval(
                        center_x - radius, center_y - radius,
                        center_x + radius, center_y + radius,
                        fill="#e0f2fe", outline="#0369a1", width=2
                    )
                    self.canvas.create_oval(
                        center_x - radius * 0.35, center_y - radius * 0.35,
                        center_x + radius * 0.35, center_y + radius * 0.35,
                        fill="#38bdf8", outline="#075985", width=1
                    )
                    self.canvas.create_oval(
                        center_x - radius * 0.13, center_y - radius * 0.13,
                        center_x + radius * 0.13, center_y + radius * 0.13,
                        fill="#0f172a", outline="#0f172a"
                    )
                    self.canvas.create_line(
                        center_x + radius * 0.45, center_y + radius * 0.45,
                        center_x + radius * 0.82, center_y + radius * 0.82,
                        fill="#0369a1", width=2
                    )
                elif value == 1:
                    self.canvas.create_text(
                        left + cell / 2,
                        top + cell / 2,
                        text="👾",
                        font=("Segoe UI Emoji", int(cell * 0.45)),
                        fill="#111827"
                    )


    def draw_state(self, node, title, note):
        self.canvas.delete("all")

        small = 24
        big = 44

        start_x = 45
        current_x = 240
        goal_x = 540

        self.canvas.create_text(410, 45, text=title, font=("Arial", 18, "bold"), fill="#111827")
        self.draw_grid(start_x, 160, small, START_GRID, START_POS, "S")
        self.draw_grid(current_x, 110, big, node.grid, node.pos, node.name)
        self.draw_grid(goal_x, 160, small, GOAL_GRID, None, "G")

        # Dynamic indicator text for Canvas
        if "A*" in self.current_title:
            f_value = node.cost + heuristic(node)
            stats_text = f"cost = {node.cost}, h = {heuristic(node)}, f = {f_value}"
        elif "Greedy" in self.current_title:
            stats_text = f"cost = {node.cost}, h = {heuristic(node)}"
        elif "Hill" in self.current_title:
            stats_text = f"h = {heuristic(node)}, v = {-heuristic(node)}"
        else:
            stats_text = f"cost = {node.cost}"

        # Positioned indicators cleanly below the main grid
        self.canvas.create_text(
            current_x + big * (SIZE / 2.0),
            110 + big * SIZE + 35,
            text=stats_text,
            font=("Arial", 12, "bold"),
            fill="#374151"
        )

    def show_solution_step(self, index):
        if index >= len(self.solution_path):
            self.show_final_result()
            return

        node = self.solution_path[index]
        action_text = f"Đi {node.action}" if node.action else "Bắt đầu"
        self.draw_state(
            node,
            f"{self.current_title} - Di chuyển bước {index}",
            f"Hành động: {action_text}"
        )

        self.after_id = self.after(STEP_DELAY, lambda: self.show_solution_step(index + 1))

    def show_random_restart_step(self, index):
        if index >= len(self.display_records):
            self.show_final_result()
            return

        record = self.display_records[index]
        section = record.get("section", "")
        title = f"{self.current_title} - bước {index + 1}"
        if section:
            title = section

        self.draw_state(
            record["show_node"],
            title,
            record["note"]
        )

        self.after_id = self.after(STEP_DELAY, lambda: self.show_random_restart_step(index + 1))

    def show_record_step(self, index):
        if index >= len(self.display_records):
            if self.was_record_limited:
                self.process_text.config(state="normal")
                self.process_text.insert("end", f"\nChi hien thi {MAX_SHOW_RECORDS} buoc dau de tranh dung giao dien.\n")
                self.process_text.config(state="disabled")
            self.show_final_result()
            return

        record = self.display_records[index]
        self.draw_state(
            record["show_node"],
            f"{self.current_title} - bước {index + 1}",
            record["note"]
        )
        self.add_process_row(record)

        self.after_id = self.after(STEP_DELAY, lambda: self.show_record_step(index + 1))

    def show_final_result(self):
        self.after_id = None

        if self.goal_node is None:
            if self.search_status == "cutoff":
                self.set_result("Chưa tìm thấy lời giải trong giới hạn độ sâu đã chọn.")
            else:
                self.set_result("Không tìm thấy lời giải.")
            return

        path = path_to_root(self.goal_node)
        node_path = " -> ".join(node.name for node in path)
        action_path = " -> ".join(node.action for node in path[1:]) if len(path) > 1 else "(không có)"
        found_goal = self.search_status == "success" and is_goal(self.goal_node.grid)

        if found_goal:
            title = "Tìm thấy lời giải bằng " + self.current_title
            path_label = "Đường đi node: "
        else:
            title = "Không tìm thấy lời giải bằng " + self.current_title
            path_label = "Đường đi tốt nhất đã thử: "

        result = (
            title + "\n"
            + path_label + node_path + "\n"
            + "Hành động: " + action_path + "\n"
            + "Tổng cost: " + str(self.goal_node.cost) + "\n\n"
            + "Trạng thái cuối:\n" + matrix_text(self.goal_node.grid, self.goal_node.pos)
        )
        self.set_result(result)


try:
    app.destroy()
except:
    pass

app = VacuumApp()
app.mainloop()
